In [2]:
!pip install sentence_transformers
%pip install pyarrow
%pip install --use-pep517 annoy
%pip install tensorflow==2.14.0
%pip install pydot
%pip install numpy==1.24.4


# Core ML / NLP
tensorflow==2.14.0
numpy==1.24.4
sentence_transformers==2.2.2
annoy==1.19.0
pyarrow==12.0.1

# Visualization
matplotlib==3.7.1
seaborn==0.12.2
pydot==1.4.2

# Data processing
pandas==2.1.1
scikit-learn==1.3.0

# Jupyter environment
jupyter==1.0.0
notebook==6.5.4
ipykernel==6.27.0

  Using cached tensorflow-2.14.0-cp311-cp311-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (3.3 kB)
  Using cached numpy-1.24.4-cp311-cp311-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (5.6 kB)
  Using cached sentence-transformers-2.2.2.tar.gz (85 kB)
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.1 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 39.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scipy to determine which version is 

In [3]:
import os

for root, dirs, files in os.walk("."):
    for name in files:
        print(os.path.join(root, name))

./query_1.csv
./query_2.csv
./query_model.tree
./data_cleaning.ipynb
./Dockerfile
./product.tree
./neighbors.csv
./product_model.tree
./data_training.ipynb
./query_embeddings.csv
./experimenting.ipynb
./product_embeddings.csv
./product_2.csv
./df_results_full.csv
./product_1.csv
./dataset_mini.csv
./product_ids.csv
./.gitattributes
./query.tree
./df_query_test_top50.csv
./Retry/query_7.csv
./Retry/product_113.csv
./Retry/product_107.csv
./Retry/product_82.csv
./Retry/product_96.csv
./Retry/product_41.csv
./Retry/product_55.csv
./Retry/product_69.csv
./Retry/product_68.csv
./Retry/product_54.csv
./Retry/product_40.csv
./Retry/product_97.csv
./Retry/product_83.csv
./Retry/product_106.csv
./Retry/product_112.csv
./Retry/query_6.csv
./Retry/query_4.csv
./Retry/product_104.csv
./Retry/product_110.csv
./Retry/product_138.csv
./Retry/product_95.csv
./Retry/product_81.csv
./Retry/product_56.csv
./Retry/product_42.csv
./Retry/product_43.csv
./Retry/product_57.csv
./Retry/product_80.csv
./Retry/

# Loading Query and Product Dataset

In [4]:
import pandas as pd

df_queries_table = pd.read_parquet('./shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_reduced_queries_table = df_queries_table[df_queries_table["small_version"] == 1]

df_products_table = pd.read_parquet('./shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_metaData_table = pd.read_csv("./shopping_queries_dataset/shopping_queries_dataset_sources.csv")

In [4]:
df_reduced_queries_table.head()

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
16,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train
17,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train
18,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train
19,19,!awnmower tires without rims,1,B07P4CF3DP,us,S,1,1,train
20,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train


In [5]:
df_products_table.head()

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),NaN,NaN,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,NaN,NaN,11 Degrees,NaN,es
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,NaN,NaN,11 Degrees,Negro (,es


In [5]:
df_joined = pd.merge(
    df_reduced_queries_table.head(),
    df_products_table,
    how="left",
    on=["product_id", "product_id"]
)

In [7]:
df_joined.head()

,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale_y
0,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch,us
1,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,Please check your existing tire Sidewall for t...,MaxAuto,NaN,us
2,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...,NaN,[QUALITY]: Hardened Steel-Iron construction wi...,Neiko,NaN,us
3,19,!awnmower tires without rims,1,B07P4CF3DP,us,S,1,1,train,2PK 13x5.00-6 13x5.00x6 13x5x6 13x5-6 2PLY Tur...,"Tire Size: 13 x 5.00 - 6 Axle: 3/4"" inside dia...",NaN,Russo,NaN,us
4,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...,No fuss. Just take off your old assembly and r...,Tire size:15x6.00-6 Ply: 4 Tubeless\n6x4.5 Whe...,Antego Tire & Wheel,Husqvarna Silver,us


# Helper functions for dimensionality reduction

In [6]:
import numpy as np

def reshape_array(input_array, d):
    k, _ = input_array.shape
    new_array = np.zeros((k, d))
    
    for i in range(k):
        for j in range(d):
            start_idx = j * (384 // d)
            end_idx = (j + 1) * (384 // d) if j < (d - 1) else 768
            chunk = input_array[i, start_idx:end_idx]
            new_array[i, j] = np.mean(chunk)
    
    return new_array


def find_embeddings(lst_product_title,i, maxlen):
    # Define a list of sentences
    sentences = list(lst_product_title)[i:i+step]
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)

    return reshape_array(np.array(sentence_embeddings), product_dim)

# Loading sentence transformer

In [8]:
from sentence_transformers import SentenceTransformer, util

# Load a pre-trained model (you can choose from various models like BERT, RoBERTa, etc.)
model = SentenceTransformer('all-MiniLM-L6-v2')

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.11/site-packages/huggingface_hub/__init__.py)

In [ ]:
# Create sentence embeddings for your sentences
sentences = ["This is an example sentence.", "Handling unknown words in embeddings is important."]
embeddings = model.encode(sentences, convert_to_tensor=True)

# The 'embeddings' variable now contains the sentence embeddings as PyTorch tensors
print(embeddings.shape)

reduced_embeddings = reshape_array(np.array(embeddings), 12)

print(reduced_embeddings.shape)
print(reduced_embeddings)


# Data Cleaning

In [ ]:
sample_size = 2000
df_queries_dataset_mini = df_reduced_queries_table.sample(sample_size, random_state = 42).reset_index(drop=True)
df_queries_dataset_mini.shape

In [ ]:
product_cols = ['product_title', 'product_description', 'product_bullet_point','product_brand','product_color','product_id']
df_products_dataset_mini = pd.merge(df_products_table[product_cols].drop_duplicates(), df_queries_dataset_mini[['product_id']].drop_duplicates() ,on = ['product_id'])
df_products_dataset_mini.shape

In [ ]:
null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

In [ ]:
df_products_dataset_mini['product_title'] = df_products_dataset_mini['product_title'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_description'] = df_products_dataset_mini['product_description'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_bullet_point'] = df_products_dataset_mini['product_bullet_point'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_brand'] = df_products_dataset_mini['product_brand'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_color'] = df_products_dataset_mini['product_color'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_queries_dataset_mini['query'] = df_queries_dataset_mini['query'].apply(lambda x : str(x).lower() if pd.notna(x) else '')

In [ ]:
null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

# Query and Product embeddings creation

In [ ]:
queries = list(df_queries_dataset_mini['query'].unique())

step = 1000
query_dim = 32

cols = ['q' + str(x) for x in list(range(0, query_dim))] + ['query']
cnt = 0

for i in range(0,len(queries),step):
    
    cnt += 1
    # Define a list of sentences
    sentences = list(queries)[i:i+step]

    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    sentence_embeddings = reshape_array(np.array(sentence_embeddings), query_dim)
    
    df_tmp = pd.DataFrame(np.concatenate((sentence_embeddings, np.array(sentences).reshape(-1,1)), axis=1))
    
    df_tmp.columns = cols
    
    df_tmp.to_csv(
        f'query_{cnt}.csv', header = True, index = False)
    
#     if cnt == 2:
#         break
        
    print(i)

In [ ]:
lst_product_title = list(df_products_dataset_mini['product_title'])
lst_product_description = list(df_products_dataset_mini['product_description'])
lst_product_bullet_point = list(df_products_dataset_mini['product_bullet_point'])
lst_product_brand = list(df_products_dataset_mini['product_brand'])
lst_product_color = list(df_products_dataset_mini['product_color'])
lst_product_id = list(df_products_dataset_mini['product_id'])

In [ ]:

step = 2000
product_dim = 32

cols = ['p' + str(x) for x in list(range(0, product_dim * 5))] + ['product_id']\

cnt = 0
for i in range(0,len(lst_product_title),step):
    cnt += 1
    product_title_embed = find_embeddings(lst_product_title, i, "s")
    product_description_embed = find_embeddings(lst_product_description, i, "")
    product_bullet_point_embed = find_embeddings(lst_product_bullet_point, i, "")
    product_brand_embed = find_embeddings(lst_product_brand, i, "")
    product_color_embed = find_embeddings(lst_product_color, i, "")
    
    # Concatenate arrays column-wise and reshape lst_product_id to (x, 1)
    df_tmp = pd.DataFrame(np.concatenate((
        product_title_embed,
        product_description_embed,
        product_bullet_point_embed,
        product_brand_embed,
        product_color_embed,
        np.array(lst_product_id[i:i + step]).reshape(-1, 1)
    ), axis=1))
    
    
    df_tmp.columns = cols
    df_tmp.to_csv(f'product_{cnt}.csv', header = True, index = False)
    
#     if (cnt == 2):
#         break
    print(i)

In [ ]:
import pandas as pd
import glob

# Get a list of CSV files that start with "product_"
file_list = glob.glob('product_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_product_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_product_df.to_csv('product_embeddings.csv', index=False)

print(concatenated_product_df.shape)

concatenated_product_df.head()

In [ ]:
# Get a list of CSV files that start with "product_"
file_list = glob.glob('query_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_query_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_query_df.to_csv('query_embeddings.csv', index=False)

print(concatenated_query_df.shape)

concatenated_query_df.head()

In [ ]:
df_queries_dataset_mini  = pd.merge(df_queries_dataset_mini, concatenated_query_df, on = 'query')
df_queries_dataset_mini  = pd.merge(df_queries_dataset_mini, concatenated_product_df, on = 'product_id')

print(df_queries_dataset_mini.shape)

df_queries_dataset_mini.head()

df_queries_dataset_mini.to_csv('dataset_mini.csv', header = True, index = False)

In [12]:
df = pd.read_csv("./dataset_mini.csv")
df.head()


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,q0,...,p152,p153,p154,p155,p156,p157,p158,p159,product_title,pid
0,2542564,椅子 お尻が痛くならない,127630,B08ZN6J5YD,jp,E,1,1,train,0.027876,...,0.016296,0.009755,0.008368,0.003349,0.010597,-0.011316,-0.028271,0.028668,NaN,NaN
1,1620569,portatiles apple macbook,82561,B08SK63434,es,S,1,1,train,0.004671,...,-0.029393,-0.010292,0.004003,0.025400,-0.020574,0.000114,-0.012392,0.051901,NaN,NaN
2,696942,dokudami tea,34655,B00712N6II,us,S,1,1,train,-0.009718,...,-0.008909,0.007664,-0.001566,0.019790,0.004879,0.017227,-0.033422,0.021500,NaN,NaN
3,696942,dokudami tea,34655,B00712N6II,us,S,1,1,train,-0.009718,...,-0.008909,0.007664,-0.001566,0.019790,0.004879,0.017227,-0.033422,0.021500,NaN,NaN
4,429009,butcher block cabinet,20819,B07XPHHW1X,us,I,1,1,train,-0.035218,...,-0.008909,0.007664,-0.001566,0.019790,0.004879,0.017227,-0.033422,0.021500,NaN,NaN


In [14]:
df_product_embedding = pd.merge(
    pd.read_csv('./product_embeddings.csv'),
    df_products_table[['product_id','product_title']],
    on = ['product_id']
).reset_index(drop=True)

# Defragment and add pid
df_product_embedding = df_product_embedding.copy()
df_product_embedding['pid'] = np.arange(len(df_product_embedding))

# Show top rows
display(df_product_embedding.head())


df_query_embedding = pd.read_csv('./query_embeddings.csv')
df_query_embedding['qid'] = range(0, df_query_embedding.shape[0])


/tmp/ipykernel_125/1220518210.py:2: DtypeWarning: Columns (161) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('./product_embeddings.csv'),


,p0,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p153,p154,p155,p156,p157,p158,p159,product_title_x,pid,product_title_y
0,0.000706,0.001991,-0.005440,-0.008086,-0.002531,-0.010276,0.024469,0.010326,-0.014985,0.026126,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,Super Star Cream Peroxide Developer 40 Volume ...
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Super Star Cream Peroxide Developer 40 Volume ...,1,Super Star Cream Peroxide Developer 40 Volume ...
2,0.001253,0.023414,0.000200,-0.010663,-0.022919,-0.016470,0.001254,0.004667,0.004675,-0.008642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,SUPER STAR 40v Stabilized Crystal Clear Liquid...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SUPER STAR 40v Stabilized Crystal Clear Liquid...,3,SUPER STAR 40v Stabilized Crystal Clear Liquid...
4,0.003040,0.000685,0.005092,0.000559,0.001823,-0.008784,0.007632,0.020714,-0.016272,-0.000447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,Schwarzkopf Professional Blonde Me Premium Dev...


In [ ]:
df_product_embedding.head()

In [ ]:
print(df_query_embedding.shape, df_product_embedding.shape)

In [ ]:
query_tower_input_dim = 32
product_tower_input_dim = (32*5)
from annoy import AnnoyIndex

q = AnnoyIndex(query_tower_input_dim, 'euclidean')
mp_query_dict = {}


for ix,row in df_query_embedding.iterrows():
    mp_query_dict[row['qid']] = row['query']
    
    key = int(row['qid'])
    vec = list(row[['q'+str(x) for x in list(range(query_tower_input_dim))]])
    
    #     print(key,vec)
    q.add_item(key,vec)


q.build(100) # 100 trees
q.save('query.tree')

top_k = 20
mat = []
for ix,row in df_query_embedding.iterrows():
    item = row['query']
    mat.append([item] + [mp_query_dict[x] for x in q.get_nns_by_item(row['qid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['query_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

print(cols)

df_neighbors1 = pd.DataFrame(mat, columns = cols)

display(df_neighbors1.head(50))

In [ ]:
p = AnnoyIndex(product_tower_input_dim, 'dot')
mp_product_dict = {}

for ix,row in df_product_embedding.iterrows():
    mp_product_dict[int(row['pid'])] = row['product_title']
    
    key = int(row['pid'])
    vec = list(row[['p'+str(x) for x in list(range(product_tower_input_dim))]])
    
#     print(key,vec)
    p.add_item(key,vec)


p.build(100) # 100 trees
p.save('product.tree')


p = AnnoyIndex(product_tower_input_dim,  'euclidean')
p.load('product.tree')



top_k = 20
mat = []
for ix,row in df_product_embedding.iterrows():
    item = row['product_title']
    mat.append([item] + [mp_product_dict[x] for x in p.get_nns_by_item(row['pid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['product_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

print(cols)

df_neighbors2 = pd.DataFrame(mat, columns = cols)

display(df_neighbors2.head(200))

# Main dataset editing 

In [9]:
df_mini = pd.read_csv("./dataset_mini.csv")
df_mini.shape
df_mini.esci_label.value_counts()
df_mini = df_mini[(df_mini.esci_label == 'E') | (df_mini.esci_label == 'I')]
df_mini.shape

(3011, 203)

In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
import transformers
from tensorflow.keras.utils import plot_model


from tensorflow.keras.layers import Input, Embedding, Concatenate, Dense, Flatten, Dot, Reshape, GlobalMaxPooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint



import keras.backend as K
from keras.layers import Input, Dense, Embedding, Flatten, Lambda, Dot
from keras.models import Model
import numpy as np
import torch

In [11]:
print([p for p in df_product_embedding.columns])

print([p for p in df_query_embedding.columns])

query_tower_cols = ['q0', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 
                                  'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 
                                  'q17', 'q18', 'q19', 'q20', 'q21', 'q22', 'q23', 'q24', 
                                  'q25', 'q26', 'q27', 'q28', 'q29', 'q30', 'q31']

product_tower_cols = ['p0', 'p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8', 'p9', 'p10',
                      'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20',
                      'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27', 'p28', 'p29', 'p30',
                      'p31', 'p32', 'p33', 'p34', 'p35', 'p36', 'p37', 'p38', 'p39', 'p40',
                      'p41', 'p42', 'p43', 'p44', 'p45', 'p46', 'p47', 'p48', 'p49', 'p50',
                      'p51', 'p52', 'p53', 'p54', 'p55', 'p56', 'p57', 'p58', 'p59', 'p60',
                      'p61', 'p62', 'p63', 'p64', 'p65', 'p66', 'p67', 'p68', 'p69', 'p70',
                      'p71', 'p72', 'p73', 'p74', 'p75', 'p76', 'p77', 'p78', 'p79', 'p80',
                      'p81', 'p82', 'p83', 'p84', 'p85', 'p86', 'p87', 'p88', 'p89', 'p90',
                      'p91', 'p92', 'p93', 'p94', 'p95', 'p96', 'p97', 'p98', 'p99', 'p100',
                      'p101', 'p102', 'p103', 'p104', 'p105', 'p106', 'p107', 'p108', 'p109', 'p110',
                      'p111', 'p112', 'p113', 'p114', 'p115', 'p116', 'p117', 'p118', 'p119', 'p120',
                      'p121', 'p122', 'p123', 'p124', 'p125', 'p126', 'p127', 'p128', 'p129', 'p130',
                      'p131', 'p132', 'p133', 'p134', 'p135', 'p136', 'p137', 'p138', 'p139', 'p140',
                      'p141', 'p142', 'p143', 'p144', 'p145', 'p146', 'p147', 'p148', 'p149', 'p150',
                      'p151', 'p152', 'p153', 'p154', 'p155', 'p156', 'p157', 'p158', 'p159'
                     ]

NameError: name 'df_product_embedding' is not defined

In [ ]:
df_mini['binary_label'] = df_mini['esci_label'].apply(lambda x: 1 if x == 'E' else 0)
df_mini.head()
# Split the dataset into training and validation
train_data = df_mini[df_mini['split'] != 'test']
val_data = df_mini[df_mini['split'] == 'test']

train_labels = np.array(train_data['binary_label'])
val_labels = np.array(val_data['binary_label'])

train_labels = train_labels.astype('float32')
val_labels = val_labels.astype('float32')

# Prepare input data for training and validation
train_inputs = [
    np.array(train_data[query_tower_cols]),
    np.array(train_data[product_tower_cols])
]

val_inputs = [
    np.array(val_data[query_tower_cols]),
    np.array(val_data[product_tower_cols])
]

In [ ]:
print(query_tower_input_dim, product_tower_input_dim)

In [ ]:
embedding_dim = 16


# Input layers for tokenized sequences
input_query = Input(shape=(query_tower_input_dim,), name='input_query')
final_query_embedding = Dense(embedding_dim, activation='linear', name='embedding_layer_query')(input_query)
normalized_query = Lambda(lambda x: tf.keras.backend.l2_normalize(x, axis=-1), name='normalize_query')(final_query_embedding)

input_product = Input(shape=(product_tower_input_dim,), name='input_product')
final_product_embedding = Dense(embedding_dim, activation='linear', name='embedding_layer_product')(input_product)
normalized_product = Lambda(lambda x: tf.keras.backend.l2_normalize(x, axis=-1), name='normalize_product')(final_product_embedding)

cosine_similarity = Dot(axes=1, normalize=True, name='cosine_similarity')([normalized_product, normalized_query])

# Build your model
model = Model(inputs=[input_query, input_product], outputs=cosine_similarity)


# Summary of the model
model.summary()

plot_model(model)


In [ ]:
# Define a model checkpoint callback to save the best model weights during training
checkpoint_path = './saved_models/best_model.keras'
model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_loss',  # Monitor validation loss
    verbose=1,
    save_best_only=True,  # Save only the best model
    mode='min'  # Minimize the validation loss
)

# Compile the model with 'binary_crossentropy' loss as a string
# model.compile(optimizer=Adam(lr=0.001), loss='mean_squared_error', metrics=['accuracy'])

model.compile(
    optimizer=Adam(learning_rate=0.001),  # <-- use learning_rate, not lr
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Fit the model with model checkpoint callback
history = model.fit(
    # Training data and parameters
    train_inputs, train_labels, epochs=100, batch_size=64, 
    validation_data=(val_inputs, val_labels),
    callbacks=[model_checkpoint],
#     class_weight=dict(enumerate(class_weights))
    # Giving positive class more importance to positive labels to ensure precision is high.
    class_weight=dict(enumerate([1,2]))
                   )


model.load_weights(checkpoint_path)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(checkpoint_path)

# Optional: optimize for size/performance
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save to file
with open("query_product_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

In [ ]:
product_model = Model(inputs=[input_query, input_product],
                         outputs=model.get_layer('normalize_product').output
                   )

input_data_product = [
    np.array([np.array([0] * query_tower_input_dim)] * df_product_embedding.shape[0]),
    np.array(df_product_embedding[product_tower_cols])
]

product_embeddings = product_model.predict(input_data_product)
len(product_embeddings)

In [ ]:
df_product_embeddings_model = pd.DataFrame(product_embeddings, columns = [f'p{x}'for x in range(embedding_dim)] )
df_product_embedding.head()
df_product_embeddings_model['product_id'] = df_product_embedding['product_id']
df_product_embeddings_model['product_title'] = df_product_embedding['product_title']
df_product_embeddings_model['pid'] = df_product_embedding['pid']



In [ ]:
# pm = AnnoyIndex(embedding_dim, 'euclidean')
pm = AnnoyIndex(embedding_dim, 'dot')
mp_product_dict = {}

for ix,row in df_product_embeddings_model.iterrows():
    mp_product_dict[int(row['pid'])] = row['product_title']
    
    key = int(row['pid'])
    vec = list(row[['p'+str(x) for x in list(range(embedding_dim))]])
    
#     print(key,vec)
    pm.add_item(key,vec)


pm.build(100) # 100 trees
pm.save('product_model.tree')

# pm = AnnoyIndex(embedding_dim,  'euclidean')
pm = AnnoyIndex(embedding_dim,  'dot')
pm.load('product_model.tree')

top_k = 20
mat = []
for ix,row in df_product_embeddings_model.iterrows():
    item = row['product_title']
    mat.append([item] + [mp_product_dict[x] for x in pm.get_nns_by_item(row['pid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['product_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

print(cols)

df_neighbors3 = pd.DataFrame(mat, columns = cols)

display(df_neighbors3.head(50))